# R21 Resting-State Randomise Results

Inspect every **cluster-extent corrected** randomise result on MNI anatomy and summarize the corresponding dual-regression stage-2 beta for sham, RTPJ, VLPFC, and BOTH. Each result is shown in its own interactive NiiVue viewer. TFCE results are intentionally excluded.

## 1. Install notebook dependencies

This cell installs only packages missing from the active kernel. It can be rerun safely and does not require Neurodesk.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'code' / 'check_randomise_results.py').is_file():
            return candidate
    raise FileNotFoundError('Start Jupyter from the r21-rest repository or one of its subdirectories.')

PROJECT_ROOT = find_project_root()
REQUIREMENTS = PROJECT_ROOT / 'notebooks' / 'requirements.txt'
required_imports = ('ipyniivue', 'ipywidgets', 'matplotlib', 'nibabel', 'nilearn', 'numpy', 'pandas')
missing = [name for name in required_imports if importlib.util.find_spec(name) is None]
if missing:
    print('Installing missing notebook packages:', ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(REQUIREMENTS)])
else:
    print('Notebook dependencies are already installed.')
print('Project root:', PROJECT_ROOT)

## 2. Load significant cluster-extent results

FSL corrected-p images contain `1-p`; the notebook defines the displayed region as voxels with `1-p > 0.95`.

In [ ]:
import json

import matplotlib.pyplot as plt
import nibabel as nib
from nilearn import datasets, image
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from ipyniivue import NiiVue

CORRP_THRESHOLD = 0.95
CONDITION_ORDER = ['sham', 'rtpj', 'vlpfc', 'both']
CONDITION_LABELS = ['Sham', 'RTPJ', 'VLPFC', 'BOTH']
NETWORK_LABELS = {'dmn': 'default mode network (DMN)', 'ecn': 'executive control network (ECN)', 'right-fpn': 'right frontoparietal network', 'left-fpn': 'left frontoparietal network'}
ANALYSIS_LABELS = {'0': 'data-derived ICA, automatic dimensionality', '20': 'data-derived ICA, 20 components', 'smith09': 'direct Smith09 RSN map'}
SUMMARY = PROJECT_ROOT / 'derivatives' / 'fsl' / 'randomise_summary' / 'task-rest_randomise_peak_summary.tsv'
if not SUMMARY.is_file():
    raise FileNotFoundError(f'Run code/check_randomise_results.py first: {SUMMARY}')

results = pd.read_csv(SUMMARY, sep='\t', dtype=str).fillna('')
if 'roi_values_tsv' not in results.columns:
    raise RuntimeError('Rerun code/check_randomise_results.py on the Linux box and push the generated ROI-value TSVs.')
significant = results.loc[(results['inference'] == 'cluster-extent') & (results['peak_gt_threshold'] == 'true') & (results['status'] == 'ok')].copy()
contrast_terms = {
    'both-minus-sham': ('BOTH', 'sham'),
    'both-minus-rtpj': ('BOTH', 'RTPJ'),
    'both-minus-vlpfc': ('BOTH', 'VLPFC'),
    'rtpj-minus-vlpfc': ('RTPJ', 'VLPFC'),
    'rtpj-minus-sham': ('RTPJ', 'sham'),
    'vlpfc-minus-sham': ('VLPFC', 'sham'),
    'both-minus-mean-rtpj-vlpfc': ('BOTH', 'average(RTPJ, VLPFC)'),
}
def signed_effect(row):
    first, second = contrast_terms[row['condition_contrast']]
    if row['direction'] == 'negative':
        first, second = second, first
    return f'{first} > {second}'

significant['effect'] = significant.apply(signed_effect, axis=1)
significant['peak_fwe_p'] = 1.0 - significant['peak_corrp'].astype(float)
if significant.empty:
    raise RuntimeError('The summary contains no complete cluster-extent maps with peak 1-p > 0.95.')
if (significant['roi_values_tsv'].str.strip() == '').any():
    raise RuntimeError('Portable ROI values are missing. Rerun code/check_randomise_results.py on Linux, then commit and push derivatives/fsl/randomise_summary.')
significant['analysis_description'] = significant['analysis'].map(ANALYSIS_LABELS)
significant['network_description'] = significant['network'].map(NETWORK_LABELS).fillna(significant['network'])
summary_table = significant[['analysis_description', 'network_description', 'component', 'effect', 'peak_fwe_p']].copy()
summary_table.columns = ['Analysis', 'Network', 'Map/component', 'Signed effect', 'Peak FWE-corrected p']
summary_table.insert(0, 'Result', range(1, len(summary_table) + 1))
print(f'Cluster-extent results available: {len(significant)}')
display(summary_table.style.format({'Peak FWE-corrected p': '{:.4f}'}).hide(axis='index'))

## 3. Define visualization and ROI-summary helpers

In [ ]:
def project_path(value):
    path = Path(value)
    return path if path.is_absolute() else PROJECT_ROOT / path

def result_image(row):
    copied = str(row.get('copied_image', '')).strip()
    source = copied or str(row['corrp_file']).strip()
    path = project_path(source)
    if not path.is_file():
        raise FileNotFoundError(path)
    return path

def result_label(row):
    return f"{row['effect']} in the {row['network_description']}"

def result_sidecar(row):
    path = project_path(row['copied_sidecar'])
    return json.loads(path.read_text()) if path.is_file() else {}

def extract_condition_values(row_index):
    row = significant.loc[row_index]
    corrp_img = nib.load(str(result_image(row)))
    corrp_data = np.asarray(corrp_img.dataobj)
    roi = np.isfinite(corrp_data) & (corrp_data > CORRP_THRESHOLD)
    if not roi.any():
        raise ValueError('Selected corrected-p map has no voxels above the threshold.')

    values_path = project_path(row['roi_values_tsv'])
    if not values_path.is_file():
        raise FileNotFoundError(f'Portable ROI-value table is not tracked locally: {values_path}')
    values = pd.read_csv(values_path, sep='\t', dtype={'participant': str, 'condition': str})
    required = {'participant', 'condition', 'stage2_beta'}
    if not required.issubset(values.columns):
        raise ValueError(f'{values_path} is missing columns: {sorted(required.difference(values.columns))}')
    values['stage2_beta'] = pd.to_numeric(values['stage2_beta'], errors='raise')
    values['condition'] = values['condition'].str.lower()
    counts = values.groupby('participant')['condition'].nunique()
    complete = counts[counts == 4].index
    values = values.loc[values['participant'].isin(complete)].copy()
    values['condition'] = pd.Categorical(values['condition'], CONDITION_ORDER, ordered=True)
    return values, corrp_img, roi

def plot_condition_bars(values, title):
    summary = values.groupby('condition', observed=False)['stage2_beta'].agg(['mean', 'sem', 'count']).reindex(CONDITION_ORDER)
    fig, ax = plt.subplots(figsize=(7.5, 4.6))
    colors = ['#8A8F98', '#2878B5', '#D95F59', '#3A9D6F']
    ax.bar(CONDITION_LABELS, summary['mean'], yerr=summary['sem'], capsize=5, color=colors, edgecolor='#222222', linewidth=0.8)
    ax.axhline(0, color='#555555', linewidth=0.8)
    ax.set_ylabel('Dual-regression stage-2 beta\n(mean ± SEM)')
    ax.set_title(title, fontsize=12, pad=12)
    ax.text(0.99, 0.02, f"n = {int(summary['count'].min())} participants", transform=ax.transAxes, ha='right', va='bottom', color='#555555', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    fig.tight_layout()
    display(fig)
    plt.close(fig)
    return summary

## 4. All significant results

Each warm-colored overlay contains only voxels with cluster-extent FWE-corrected `1-p > 0.95` (corrected `p < 0.05`). The overlay encodes corrected significance, not connectivity magnitude. Use the NiiVue controls to change slice position, orientation, opacity, and rendering mode.

In [ ]:
CACHE_DIR = PROJECT_ROOT / 'derivatives' / 'fsl' / 'randomise_summary' / '.notebook_cache'
CACHE_DIR.mkdir(exist_ok=True)
mni_img = datasets.load_mni152_template(resolution=2)
mni_path = Path(mni_img.get_filename()) if mni_img.get_filename() else CACHE_DIR / 'MNI152_T1_2mm.nii.gz'
if not mni_path.is_file():
    nib.save(mni_img, mni_path)

def thresholded_map_path(row, corrp_img, roi):
    source = result_image(row)
    destination = CACHE_DIR / source.name.replace('_stat-corrp_statmap.nii.gz', '_desc-thresholded_statmap.nii.gz')
    thresholded = image.new_img_like(corrp_img, np.where(roi, np.asarray(corrp_img.dataobj), 0.0), copy_header=True)
    nib.save(thresholded, destination)
    return destination

for result_number, (row_index, row) in enumerate(significant.iterrows(), start=1):
    values, corrp_img, roi = extract_condition_values(row_index)
    sidecar = result_sidecar(row)
    corrected_p = 1.0 - float(row['peak_corrp'])
    n_participants = values['participant'].nunique()
    map_label = f"map {int(row['component'])}" if row['analysis'] == 'smith09' else f"component {int(row['component'])}"
    display(Markdown(
        f"### Result {result_number}: {result_label(row)}\n\n"
        f"- **Network definition:** {row['analysis_description']}, {map_label}.\n"
        f"- **Tested contrast:** `{row['condition_contrast']}`, {row['design_contrast']} ({row['direction']}); this is **{row['effect']}**.\n"
        f"- **Inference:** one-sample randomise, {int(sidecar.get('NumberOfPermutations', 5000)):,} permutations, cluster-forming *t* = {float(sidecar.get('ClusterFormingTThreshold', 3.1)):.1f}.\n"
        f"- **Corrected result:** peak FWE-corrected *p* = {corrected_p:.4f}; {int(roi.sum())} suprathreshold voxels.\n"
        f"- **Bar plot:** mean stage-2 beta ± SEM across {n_participants} participants within the displayed cluster."
    ))
    overlay_path = thresholded_map_path(row, corrp_img, roi)
    viewer = NiiVue()
    viewer.load_volumes([
        {'path': str(mni_path), 'name': 'MNI152 T1 2 mm', 'colormap': 'gray', 'opacity': 1.0},
        {'path': str(overlay_path), 'name': row['effect'], 'colormap': 'red', 'opacity': 0.80, 'cal_min': CORRP_THRESHOLD, 'cal_max': 1.0},
    ])
    display(viewer)
    plot_condition_bars(values, result_label(row))

## Interpretation note

The bars show participant-level means and SEM extracted from the stage-2 beta maps. Because the displayed region was selected from the same group contrast, these values are a descriptive visualization of the result, not an independent ROI test. Confirmatory ROI inference requires an independently defined mask or held-out data.